<a href="https://www.kaggle.com/code/alexvmt/terainet-inference-example?scriptVersionId=247707904" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# TeraiNet inference example

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py) for the general procedure

## Setup

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [1]:
!pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.5 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.0
    Uninstalling ml_dtypes-0.5.0:
      Successfully uninstalled ml_dtypes-0.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.4.35 requires ml-dtypes>=0.4.0, but you have ml-dtypes 0.3.2 which is incompatible.


In [2]:
import os
import yaml
import random
import shutil
import pandas as pd
from pathlib import Path

import kimm
import tensorflow as tf
from keras import saving

### Utilities

In [3]:
def copy_random_images(src_dir, dst_dir, n, seed=42):
    """
    Copies n random images from src_dir to dst_dir.

    Args:
        src_dir (str): Source directory containing images.
        dst_dir (str): Destination directory to copy images into.
        n (int): Number of images to copy.
        seed (int): Random seed for reproducibility.
    """
    # Ensure target directory exists
    os.makedirs(dst_dir, exist_ok=True)

    # List all files in the source directory
    all_files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]

    # Check if n is greater than available files
    if n > len(all_files):
        raise ValueError(f"Requested {n} images, but only {len(all_files)} available in source directory.")

    # Randomly sample n files
    random.seed(seed)
    selected_files = random.sample(all_files, n)

    # Copy each file to the target directory
    for filename in selected_files:
        src_path = os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)
        shutil.copy2(src_path, dst_path)

    print(f"Copied {n} random images from '{src_dir}' to '{dst_dir}'.")

## Prepare images

In [4]:
copy_random_images("../input/preprocess-images/terainet_images/test2/class_1", "../images", 10)

Copied 10 random images from '../input/preprocess-images/terainet_images/test2/class_1' to '../images'.


In [5]:
img_generator = tf.keras.preprocessing.image_dataset_from_directory(
    "../images", 
    labels=None,
    label_mode=None,
    batch_size=8, 
    image_size=(224, 224),
    shuffle=False
)

Found 10 files.


## Predict

In [6]:
model = saving.load_model("../input/train-and-evaluate-terainet/model.keras", compile=False)

In [7]:
preds = model.predict(img_generator)

2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 7s/step


In [8]:
preds

array([[0.8380921 , 0.01930532, 0.02411051, 0.01074985, 0.01583928,
        0.01876356, 0.0116556 , 0.02094849, 0.02741486, 0.01312049],
       [0.02425872, 0.01655218, 0.03171601, 0.6747775 , 0.04991604,
        0.09536092, 0.01645462, 0.03218096, 0.00672195, 0.05206098],
       [0.6976975 , 0.03045564, 0.07910504, 0.02524918, 0.04490445,
        0.03997582, 0.01651551, 0.01778198, 0.02397633, 0.02433849],
       [0.8112411 , 0.01984998, 0.02115559, 0.01794998, 0.0249802 ,
        0.03154941, 0.02464576, 0.01624035, 0.01597141, 0.01641617],
       [0.7346647 , 0.01480038, 0.08741897, 0.02281096, 0.03066213,
        0.02725622, 0.04286971, 0.01431758, 0.010711  , 0.01448843],
       [0.8057046 , 0.02298744, 0.02497264, 0.02121506, 0.024504  ,
        0.01712161, 0.02366268, 0.02204646, 0.01665653, 0.02112889],
       [0.04180597, 0.02016869, 0.03934361, 0.18289427, 0.61903447,
        0.01692739, 0.03795557, 0.01188495, 0.00556668, 0.02441834],
       [0.02222991, 0.04452099, 0.3300904

## Post-processing

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py)

In [9]:
with open("../input/train-and-evaluate-terainet/class_list.yaml", "r") as file:
    class_map = yaml.safe_load(file)
class_map

{'1': 'tiger',
 '10': 'bird',
 '2': 'leopard',
 '3': 'black_bear',
 '4': 'other_carnivores',
 '5': 'deer',
 '6': 'wild_boar',
 '7': 'buffalo',
 '8': 'rhino',
 '9': 'elephant'}

In [10]:
inv_class = {v: k for k, v in class_map.items()}
inv_class

{'tiger': '1',
 'bird': '10',
 'leopard': '2',
 'black_bear': '3',
 'other_carnivores': '4',
 'deer': '5',
 'wild_boar': '6',
 'buffalo': '7',
 'rhino': '8',
 'elephant': '9'}

In [11]:
file_paths = img_generator.file_paths
filenames = list(map(lambda x : Path(x).name, file_paths))
labels = list(map(lambda x : Path(x).parent.name, file_paths))

In [12]:
class_ids = sorted(inv_class.values())
class_names = [class_map.get(i,i)  for i in class_ids]
pred_df = pd.DataFrame(preds, columns=class_ids)
pred_df.head()

,1,10,2,3,4,5,6,7,8,9
0,0.838092,0.019305,0.024111,0.010750,0.015839,0.018764,0.011656,0.020948,0.027415,0.013120
1,0.024259,0.016552,0.031716,0.674778,0.049916,0.095361,0.016455,0.032181,0.006722,0.052061
2,0.697698,0.030456,0.079105,0.025249,0.044904,0.039976,0.016516,0.017782,0.023976,0.024338
3,0.811241,0.019850,0.021156,0.017950,0.024980,0.031549,0.024646,0.016240,0.015971,0.016416
4,0.734665,0.014800,0.087419,0.022811,0.030662,0.027256,0.042870,0.014318,0.010711,0.014488


In [13]:
file_series = pd.Series(filenames)
label_series = pd.Series(labels)
pred_df.insert(0, "filename", file_series, True)
pred_df.insert(1, "label", label_series, True)
pred_df = pd.melt(pred_df, id_vars=['filename', 'label'], value_vars=class_ids, var_name="class_id", value_name="prob")
pred_df["class_name"] = pred_df["class_id"].replace(class_map)
pred_df["class_rank"] = pred_df.groupby("filename")["prob"].rank("average", ascending=False)
pred_df.head(10)

,filename,label,class_id,prob,class_name,class_rank
0,class_1_test2_118-0.jpg,images,1,0.838092,tiger,1.0
1,class_1_test2_135-0.jpg,images,1,0.024259,tiger,7.0
2,class_1_test2_143-0.jpg,images,1,0.697698,tiger,1.0
3,class_1_test2_185-0.jpg,images,1,0.811241,tiger,1.0
4,class_1_test2_19-0.jpg,images,1,0.734665,tiger,1.0
5,class_1_test2_210-0.jpg,images,1,0.805705,tiger,1.0
6,class_1_test2_283-0.jpg,images,1,0.041806,tiger,3.0
7,class_1_test2_3-0.jpg,images,1,0.022230,tiger,8.0
8,class_1_test2_310-0.jpg,images,1,0.845792,tiger,1.0
9,class_1_test2_75-0.jpg,images,1,0.712407,tiger,1.0


In [14]:
pred_df = pred_df[pred_df["class_rank"] == 1.0]
pred_df = pred_df.drop(["label", "class_rank"], axis=1)
pred_df

,filename,class_id,prob,class_name
0,class_1_test2_118-0.jpg,1,0.838092,tiger
2,class_1_test2_143-0.jpg,1,0.697698,tiger
3,class_1_test2_185-0.jpg,1,0.811241,tiger
4,class_1_test2_19-0.jpg,1,0.734665,tiger
5,class_1_test2_210-0.jpg,1,0.805705,tiger
8,class_1_test2_310-0.jpg,1,0.845792,tiger
9,class_1_test2_75-0.jpg,1,0.712407,tiger
27,class_1_test2_3-0.jpg,2,0.330090,leopard
31,class_1_test2_135-0.jpg,3,0.674778,black_bear
46,class_1_test2_283-0.jpg,4,0.619034,other_carnivores
